# Merge layoff pool with no-layoff pool, labeled by source only

Same two input pools as `merge_layoff_no_layoff.ipynb` (`merged_bs_cs_fd.csv` and `merged_bs_cs_fd_no_layoff.csv`), but the label here ignores the quarter a layoff happened in entirely: every row from the layoff-tracked pool is `layoff = 1`, every row from the confirmed no-layoff pool is `layoff = 0`.

Use this when the question is "did this company ever have a layoff" rather than "did this company have a layoff in this specific quarter".

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA

LAYOFF_CSV = MERGED_DATA["MERGED_OUTPUT_CSV_PATH"]
NO_LAYOFF_CSV = MERGED_DATA["MERGED_OUTPUT_NO_LAYOFF_CSV_PATH"]
FINAL_OUTPUT_CSV = MERGED_DATA["FINAL_MERGED_SOURCE_LABELED_CSV_PATH"]

ID_COLS = ["company", "date", "quarter"]

## 1. Load both pools

In [2]:
layoff_df = pd.read_csv(LAYOFF_CSV)
no_layoff_df = pd.read_csv(NO_LAYOFF_CSV)

layoff_df["date"] = pd.to_datetime(layoff_df["date"])
no_layoff_df["date"] = pd.to_datetime(no_layoff_df["date"])

print(f"Layoff pool:    {layoff_df.shape}, companies={layoff_df['company'].nunique()}")
print(f"No-layoff pool: {no_layoff_df.shape}, companies={no_layoff_df['company'].nunique()}")

Layoff pool:    (12047, 349), companies=1941
No-layoff pool: (17983, 349), companies=3442


## 2. Clean

Drop exact duplicate rows and check that `(company, date)` is a unique key within each pool.

In [3]:
for name, df in [("layoff", layoff_df), ("no_layoff", no_layoff_df)]:
    before = len(df)
    df.drop_duplicates(inplace=True)
    dupe_keys = df.duplicated(subset=["company", "date"]).sum()
    print(
        f"{name}: dropped {before - len(df)} exact duplicate rows, "
        f"{dupe_keys} remaining duplicate (company, date) keys"
    )
    assert dupe_keys == 0, f"{name} pool has duplicate (company, date) keys"

layoff: dropped 0 exact duplicate rows, 0 remaining duplicate (company, date) keys
no_layoff: dropped 0 exact duplicate rows, 0 remaining duplicate (company, date) keys


## 3. Test: do the feature columns match?

In [4]:
layoff_feature_cols = set(layoff_df.columns) - set(ID_COLS)
no_layoff_feature_cols = set(no_layoff_df.columns) - set(ID_COLS)

only_in_layoff = sorted(layoff_feature_cols - no_layoff_feature_cols)
only_in_no_layoff = sorted(no_layoff_feature_cols - layoff_feature_cols)
shared_cols = layoff_feature_cols & no_layoff_feature_cols

print(f"Shared feature columns: {len(shared_cols)}")
print(f"Only in layoff pool ({len(only_in_layoff)}): {only_in_layoff}")
print(f"Only in no-layoff pool ({len(only_in_no_layoff)}): {only_in_no_layoff}")

dtype_conflicts = [
    c for c in shared_cols if layoff_df[c].dtype != no_layoff_df[c].dtype
]
print(f"Dtype conflicts on shared columns: {dtype_conflicts}")

Shared feature columns: 345
Only in layoff pool (1): ['bs_Restricted Common Stock']
Only in no-layoff pool (1): ['cf_Change In Dividend Payable']
Dtype conflicts on shared columns: []


## 4. Resolve column conflicts

Align both pools to the union of feature columns so a plain concat does not need to guess at missing fields — any column absent from one pool is added back as `NaN` there.

In [5]:
union_feature_cols = sorted(layoff_feature_cols | no_layoff_feature_cols)

missing_in_layoff = [c for c in union_feature_cols if c not in layoff_df.columns]
missing_in_no_layoff = [c for c in union_feature_cols if c not in no_layoff_df.columns]

layoff_df = pd.concat(
    [layoff_df, pd.DataFrame(pd.NA, index=layoff_df.index, columns=missing_in_layoff)],
    axis=1,
)
no_layoff_df = pd.concat(
    [no_layoff_df, pd.DataFrame(pd.NA, index=no_layoff_df.index, columns=missing_in_no_layoff)],
    axis=1,
)

assert set(union_feature_cols) <= set(layoff_df.columns)
assert set(union_feature_cols) <= set(no_layoff_df.columns)
print(f"Aligned both pools to {len(union_feature_cols)} feature columns")

Aligned both pools to 347 feature columns


## 5. Resolve ticker conflicts

A ticker should not live in both pools — the no-layoff pool is supposed to be a clean negative set. Any overlap is dropped from the no-layoff pool since that company's financials already exist (and will be labeled `1`) in the layoff pool.

In [6]:
overlap_tickers = sorted(set(layoff_df["company"]) & set(no_layoff_df["company"]))
print(f"Tickers present in both pools ({len(overlap_tickers)}): {overlap_tickers}")

if overlap_tickers:
    no_layoff_df = no_layoff_df.loc[~no_layoff_df["company"].isin(overlap_tickers)].copy()
    print(f"Dropped {len(overlap_tickers)} overlapping tickers from the no-layoff pool")

assert not (set(layoff_df["company"]) & set(no_layoff_df["company"])), "ticker overlap remains"

Tickers present in both pools (2): ['STNE', 'TEAD']
Dropped 2 overlapping tickers from the no-layoff pool


## 6. Build the combined dataset

Label is purely which pool a row came from: `1` for the layoff-tracked pool, `0` for the confirmed no-layoff pool. No quarter-level matching is done here.

In [7]:
layoff_df["layoff"] = 1
no_layoff_df["layoff"] = 0

layoff_df["source_pool"] = "layoff_tracked"
no_layoff_df["source_pool"] = "confirmed_no_layoff"

final_cols = ID_COLS + union_feature_cols + ["layoff", "source_pool"]

combined = pd.concat(
    [layoff_df[final_cols], no_layoff_df[final_cols]],
    ignore_index=True,
)

print(combined.shape)

(30018, 352)


C:\Users\phuon\AppData\Local\Temp\ipykernel_7312\2404030816.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  layoff_df["layoff"] = 1
C:\Users\phuon\AppData\Local\Temp\ipykernel_7312\2404030816.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  layoff_df["source_pool"] = "layoff_tracked"


## 7. Validate the merge

In [8]:
assert combined.duplicated(subset=["company", "date"]).sum() == 0, "duplicate (company, date) rows"
assert set(combined.columns) == set(final_cols)
assert combined["layoff"].isin([0, 1]).all()

print("Companies:", combined["company"].nunique())
print(combined["layoff"].value_counts())
print(combined["source_pool"].value_counts())

Companies: 5381
layoff
0    17971
1    12047
Name: count, dtype: int64
source_pool
confirmed_no_layoff    17971
layoff_tracked         12047
Name: count, dtype: int64


## 8. Save

In [9]:
combined.to_csv(FINAL_OUTPUT_CSV, index=False)
print(f"Wrote {len(combined)} rows to {FINAL_OUTPUT_CSV}")

Wrote 30018 rows to c:\Users\phuon\Desktop\osint-proj\layoff-detector\data\processed\final_merged_source_labeled_dataset.csv
